In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Identifica a raiz do projeto.
# Se o notebook estiver sendo executado dentro da pasta notebooks/, a raiz fica um nível acima.
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

# Caso o VS Code execute o notebook em outro diretório, tenta localizar a raiz pelo caminho relativo.
if not (PROJECT_DIR / "data").exists() and (Path("..") / "data").exists():
    PROJECT_DIR = Path("..").resolve()

TRAIN_CSV = PROJECT_DIR / "data" / "raw" / "train.csv"
TRAIN_IMAGES_DIR = PROJECT_DIR / "data" / "raw" / "train_images"

print("Raiz do projeto:", PROJECT_DIR)
print("Arquivo train.csv:", TRAIN_CSV)
print("Pasta train_images:", TRAIN_IMAGES_DIR)

if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {TRAIN_CSV}")

if not TRAIN_IMAGES_DIR.exists():
    raise FileNotFoundError(f"Pasta não encontrada: {TRAIN_IMAGES_DIR}")

df = pd.read_csv(TRAIN_CSV)

df.head()


In [ ]:
# Informações gerais do dataframe
print(df.info())

# Quantidade de linhas e colunas
print("Dimensão do train.csv:", df.shape)

# Conferir nomes das colunas
print("Colunas:", df.columns.tolist())

In [ ]:
class_names = {
    0: "Sem retinopatia",
    1: "Retinopatia leve",
    2: "Retinopatia moderada",
    3: "Retinopatia severa",
    4: "Retinopatia proliferativa"
}

df["diagnosis_name"] = df["diagnosis"].map(class_names)

df.head()

In [ ]:
class_distribution = (
    df["diagnosis"]
    .value_counts()
    .sort_index()
    .reset_index()
)

class_distribution.columns = ["classe", "quantidade"]
class_distribution["descricao"] = class_distribution["classe"].map(class_names)
class_distribution["percentual"] = (
    class_distribution["quantidade"] / class_distribution["quantidade"].sum() * 100
).round(2)

class_distribution

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    class_distribution["descricao"],
    class_distribution["quantidade"]
)

plt.title("Distribuição das classes na base APTOS 2019")
plt.xlabel("Classe")
plt.ylabel("Quantidade de imagens")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

plt.show()

In [ ]:
df["image_path"] = df["id_code"].apply(
    lambda x: TRAIN_IMAGES_DIR / f"{x}.png"
)

df["image_exists"] = df["image_path"].apply(lambda x: x.exists())

total_images_csv = len(df)
existing_images = int(df["image_exists"].sum())
missing_images = total_images_csv - existing_images

print("Total de registros no train.csv:", total_images_csv)
print("Imagens encontradas:", existing_images)
print("Imagens ausentes:", missing_images)

if missing_images > 0:
    print("\nAtenção: existem imagens listadas no train.csv que não foram encontradas na pasta train_images.")
    print("Verifique se a base APTOS foi copiada ou extraída completamente.")


In [ ]:
df_missing = df[df["image_exists"] == False]

df_missing

In [ ]:
RESULTS_DIR = PROJECT_DIR / "results" / "metrics"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

class_distribution.to_csv(
    RESULTS_DIR / "class_distribution.csv",
    index=False
)

print("Arquivo salvo em:", RESULTS_DIR / "class_distribution.csv")

In [ ]:
FIGURES_DIR = PROJECT_DIR / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(10, 6))

plt.bar(
    class_distribution["descricao"],
    class_distribution["quantidade"]
)

plt.title("Distribuição das classes na base APTOS 2019")
plt.xlabel("Classe")
plt.ylabel("Quantidade de imagens")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

plt.savefig(FIGURES_DIR / "class_distribution.png", dpi=300)
plt.show()

print("Gráfico salvo em:", FIGURES_DIR / "class_distribution.png")

In [ ]:
import pandas as pd
from pathlib import Path

# Reutiliza PROJECT_DIR se ele já existir; caso contrário, tenta localizar a raiz do projeto.
try:
    PROJECT_DIR
except NameError:
    CURRENT_DIR = Path.cwd()
    PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

SPLITS_DIR = PROJECT_DIR / "data" / "splits"

TRAIN_SPLIT = SPLITS_DIR / "train_split.csv"
VAL_SPLIT = SPLITS_DIR / "val_split.csv"
TEST_SPLIT = SPLITS_DIR / "test_split.csv"

for file_path in [TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {file_path}\n"
            "Execute antes: python src/prepare_splits.py"
        )

train_df = pd.read_csv(TRAIN_SPLIT)
val_df = pd.read_csv(VAL_SPLIT)
test_df = pd.read_csv(TEST_SPLIT)

print("Treino:", train_df.shape)
print("Validação:", val_df.shape)
print("Teste:", test_df.shape)

print("\nDistribuição no treino:")
print(train_df["diagnosis"].value_counts().sort_index())

print("\nDistribuição na validação:")
print(val_df["diagnosis"].value_counts().sort_index())

print("\nDistribuição no teste:")
print(test_df["diagnosis"].value_counts().sort_index())

print("\nPercentual no treino:")
print((train_df["diagnosis"].value_counts(normalize=True).sort_index() * 100).round(2))

print("\nPercentual na validação:")
print((val_df["diagnosis"].value_counts(normalize=True).sort_index() * 100).round(2))

print("\nPercentual no teste:")
print((test_df["diagnosis"].value_counts(normalize=True).sort_index() * 100).round(2))

split_summary = pd.DataFrame({
    "conjunto": ["Treinamento", "Validação", "Teste", "Total"],
    "quantidade": [len(train_df), len(val_df), len(test_df), len(train_df) + len(val_df) + len(test_df)],
    "percentual": [
        round(len(train_df) / (len(train_df) + len(val_df) + len(test_df)) * 100, 2),
        round(len(val_df) / (len(train_df) + len(val_df) + len(test_df)) * 100, 2),
        round(len(test_df) / (len(train_df) + len(val_df) + len(test_df)) * 100, 2),
        100.00
    ]
})

split_summary


In [ ]:
RESULTS_DIR = PROJECT_DIR / "results" / "metrics"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

split_summary.to_csv(
    RESULTS_DIR / "split_summary.csv",
    index=False
)

split_class_distribution = pd.concat(
    [
        train_df.assign(conjunto="Treinamento"),
        val_df.assign(conjunto="Validação"),
        test_df.assign(conjunto="Teste")
    ],
    ignore_index=True
)

split_class_distribution = (
    split_class_distribution
    .groupby(["conjunto", "diagnosis"])
    .size()
    .reset_index(name="quantidade")
)

split_class_distribution["descricao"] = split_class_distribution["diagnosis"].map(class_names)

split_class_distribution["percentual_no_conjunto"] = (
    split_class_distribution["quantidade"] /
    split_class_distribution.groupby("conjunto")["quantidade"].transform("sum") *
    100
).round(2)

split_class_distribution.to_csv(
    RESULTS_DIR / "split_class_distribution.csv",
    index=False
)

print("Resumo da divisão salvo em:", RESULTS_DIR / "split_summary.csv")
print("Distribuição por classe em cada subconjunto salva em:", RESULTS_DIR / "split_class_distribution.csv")

split_class_distribution
